# ACT's complete flow:
Current_State + Action_Sequence → Encoder → CLS_Output → Latent_Z → Decoder → Final_Actions

In [ ]:
'''
# Training: Learn action distributions given current state + demonstrated actions
encoder_input = [CLS, current_qpos, action_sequence]
# Model learns: "Given where I am now + what I should do, refine the actions"

# During execution:
encoder_input = [CLS, current_qpos, zeros/previous_actions]  
# Model can adapt based on CURRENT state, not just past history

# Example: Robot is reaching for cup, suddenly cup moves

# ACT Approach:
current_qpos = [new_arm_position]  # Current state after cup moved
# Model immediately gets updated state information
# Can adapt action sequence based on NEW current position

# Old Approach:  
state_history = [old_positions × 99, new_position × 1]
# Still heavily influenced by 99 old positions
# Takes longer to adapt to sudden environmental changes
'''

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import time
import math
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torchvision import transforms, models
from PIL import Image
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        return None, None, None, None

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def load_all_episodes(data_dir):
    """
    Load all HDF5 files from the specified directory
    Args:
        data_dir: directory containing HDF5 files
    Returns:
        combined qpos, qvel, action arrays and list of image dicts
    """
    print(f"Loading episodes from: {data_dir}")
    
    # Find all HDF5 files
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()  # Sort for consistent ordering
    
    if not hdf5_files:
        print("No HDF5 files found in the directory!")
        return None, None, None, None
    
    print(f"Found {len(hdf5_files)} HDF5 files: {hdf5_files}")
    
    all_qpos = []
    all_qvel = []
    all_action = []
    all_image_dicts = []
    
    episode_info = []
    
    for filename in hdf5_files:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")
        
        qpos, qvel, action, image_dict = load_hdf5(filepath)
        
        if qpos is not None:
            print(f"  - Episode length: {len(qpos)} timesteps")
            print(f"  - QPos shape: {qpos.shape}")
            print(f"  - Action shape: {action.shape}")
            
            all_qpos.append(qpos)
            all_qvel.append(qvel)
            all_action.append(action)
            all_image_dicts.append(image_dict)
            
            episode_info.append({
                'filename': filename,
                'length': len(qpos),
                'start_idx': sum(len(ep) for ep in all_qpos[:-1]),
                'end_idx': sum(len(ep) for ep in all_qpos)
            })
        else:
            print(f"  - Failed to load {filename}")
    
    if not all_qpos:
        print("No valid episodes loaded!")
        return None, None, None, None
    
    # Concatenate all episodes
    combined_qpos = np.concatenate(all_qpos, axis=0)
    combined_qvel = np.concatenate(all_qvel, axis=0)
    combined_action = np.concatenate(all_action, axis=0)
    
    print(f"\nCombined dataset:")
    print(f"Total episodes: {len(all_qpos)}")
    print(f"Total timesteps: {len(combined_qpos)}")
    print(f"Combined QPos shape: {combined_qpos.shape}")
    print(f"Combined Action shape: {combined_action.shape}")
    
    # Print episode breakdown
    print(f"\nEpisode breakdown:")
    for info in episode_info:
        print(f"  {info['filename']}: {info['length']} steps (indices {info['start_idx']}-{info['end_idx']-1})")
    
    return combined_qpos, combined_qvel, combined_action, all_image_dicts

# Load all episodes from the directory
data_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task1c'
qpos, qvel, action, image_dict_list = load_all_episodes(data_dir)

if qpos is None:
    print("Failed to load any data!")
    exit()

# Create DataFrames with joint headers
joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']

# Create timestamp column (assuming sequential timesteps across all episodes)
num_timesteps = len(qpos)
timestamps = np.arange(num_timesteps)

# Create qpos DataFrame
qpos_data = np.column_stack([timestamps, qpos])
qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)

# Create action DataFrame
action_data = np.column_stack([timestamps, action])
action_df = pd.DataFrame(action_data, columns=joint_headers)

print("\nDataFrame Summary:")
print("QPos DataFrame shape:", qpos_df.shape)
print("Action DataFrame shape:", action_df.shape)
print("\nQPos DataFrame head:")
print(qpos_df.head())
print("\nAction DataFrame head:")
print(action_df.head())

# Check the data ranges for each joint
print("\nQPos data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")

print("\nAction data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")

# Optional: Plot data distribution across episodes
plt.figure(figsize=(15, 10))

joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist1', 'Wrist2', 'Gripper']

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    plt.subplot(2, 3, i+1)
    plt.plot(qpos_df['timestamp'], qpos_df[joint], 'b-', alpha=0.7, label=f'QPos {joint}')
    plt.plot(action_df['timestamp'], action_df[joint], 'r-', alpha=0.7, label=f'Action {joint}')
    plt.title(f'{joint_name} Joint ({joint}) - All Episodes')
    plt.xlabel('Timestep')
    plt.ylabel('Joint Value (radians)')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.suptitle('Joint Data Across All Episodes', fontsize=16)
plt.tight_layout()
plt.show()

print("\nData loading complete! Now you can proceed with preprocessing and training.")

## Resnet34
### Position encoding : static , learning

In [ ]:
class PositionEmbeddingSine(nn.Module):
    """ACT's sinusoidal position embedding for images"""
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, channels, height, width]
        Returns:
            pos: [batch, num_pos_feats*2, height, width]
        """
        x = tensor
        # Create a mask of ones (since we don't have actual masks)
        not_mask = torch.ones_like(x[0, [0]])  # [1, height, width]
        
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos
    
class PositionEmbeddingLearned(nn.Module):
    """ACT's learnable position embedding for images"""
    def __init__(self, num_pos_feats=256):
        super().__init__()
        self.row_embed = nn.Embedding(50, num_pos_feats)
        self.col_embed = nn.Embedding(50, num_pos_feats)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.row_embed.weight)
        nn.init.uniform_(self.col_embed.weight)

    def forward(self, tensor):
        """
        Args:
            tensor: [batch, channels, height, width]
        Returns:
            pos: [batch, num_pos_feats*2, height, width]
        """
        x = tensor
        h, w = x.shape[-2:]
        i = torch.arange(w, device=x.device)
        j = torch.arange(h, device=x.device)
        x_emb = self.col_embed(i)
        y_emb = self.row_embed(j)
        pos = torch.cat([
            x_emb.unsqueeze(0).repeat(h, 1, 1),
            y_emb.unsqueeze(1).repeat(1, w, 1),
        ], dim=-1).permute(2, 0, 1).unsqueeze(0).repeat(x.shape[0], 1, 1, 1)
        return pos
    

### ACT backbone

In [ ]:
# ✅ ACT-style backbone with proper position encoding
class FrozenBatchNorm2d(torch.nn.Module):
    """ACT's frozen batch norm"""
    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def forward(self, x):
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-5
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

class ACTBackbone(nn.Module):
    """ACT-style ResNet backbone with position encoding"""
    def __init__(self, 
                 backbone_name='resnet34',
                 train_backbone=True,
                 return_interm_layers=False,
                 position_embedding_type='sine',  # 'learned' or 'sine'
                 hidden_dim=256):
        super().__init__()
        
        # ✅ Create backbone like ACT
        backbone = getattr(models, backbone_name)(pretrained=True)
        
        # Replace BatchNorm with FrozenBatchNorm (like ACT)
        if not train_backbone:
            for module in backbone.modules():
                if isinstance(module, nn.BatchNorm2d):
                    # This is simplified - ACT does this during model creation
                    module.eval()
                    for param in module.parameters():
                        param.requires_grad = False
        
        # Remove final layers
        if return_interm_layers:
            return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
        else:
            return_layers = {'layer4': "0"}
        
        from torchvision.models._utils import IntermediateLayerGetter
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        
        # Set number of channels
        self.num_channels = 512 if backbone_name in ('resnet18', 'resnet34') else 2048
        
        # ✅ Position embedding (like ACT)
        N_steps = hidden_dim // 2
        if position_embedding_type == 'learned':
            self.position_embedding = PositionEmbeddingLearned(N_steps)
        elif position_embedding_type == 'sine':
            self.position_embedding = PositionEmbeddingSine(N_steps, normalize=True)
        else:
            raise ValueError(f"Unknown position embedding: {position_embedding_type}")
    
    def forward(self, images):
        """
        Args:
            images: [batch, 3, height, width]
        Returns:
            features: dict with feature maps
            pos_embeds: dict with position embeddings
        """
        # Extract features
        features = self.body(images)
        
        # Generate position embeddings for each feature level
        pos_embeds = {}
        for name, feature_map in features.items():
            pos_embeds[name] = self.position_embedding(feature_map)
        
        return features, pos_embeds

### Extract feature 

In [ ]:
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms, models
import h5py
import os
import numpy as np

# ========================================
# INPUT PROJECTION LAYER (ACT-STYLE)
# ========================================

class ACTInputProjection(nn.Module):
    """ACT's input projection layer: projects visual features to transformer embedding dimension"""
    def __init__(self, backbone_channels, hidden_dim):
        super().__init__()
        # self.input_proj = nn.Conv2d(backbones[0].num_channels, hidden_dim, kernel_size=1)
        self.input_proj = nn.Conv2d(backbone_channels, hidden_dim, kernel_size=1)
        
    def forward(self, features):
        """
        Args:
            features: [batch, backbone_channels, H, W] (e.g., [batch, 512, H, W])
        Returns:
            projected: [batch, hidden_dim, H, W] (e.g., [batch, 256, H, W])
        """
        return self.input_proj(features)

# ========================================
# DATA LOADING FUNCTIONS
# ========================================

def load_all_images_from_episodes(data_dir, camera_names=['front']):
    """Load all images from all episodes across multiple HDF5 files"""
    print(f"Loading images from episodes in: {data_dir}")
    
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    hdf5_files.sort()
    
    if not hdf5_files:
        print("No HDF5 files found!")
        return None, None
    
    all_images = {cam_name: [] for cam_name in camera_names}
    image_episode_info = []
    
    for episode_idx, filename in enumerate(hdf5_files):
        filepath = os.path.join(data_dir, filename)
        print(f"Loading images from {filename}...")
        
        with h5py.File(filepath, 'r') as root:
            episode_start_idx = len(all_images[camera_names[0]])
            
            for cam_name in camera_names:
                if f'/observations/images/{cam_name}' in root:
                    episode_images = root[f'/observations/images/{cam_name}'][()]
                    all_images[cam_name].extend(episode_images)
                    print(f"  - {cam_name}: {len(episode_images)} images")
                else:
                    print(f"  - Warning: {cam_name} camera not found in {filename}")
            
            episode_end_idx = len(all_images[camera_names[0]])
            
            image_episode_info.append({
                'filename': filename,
                'episode_idx': episode_idx,
                'start_idx': episode_start_idx,
                'end_idx': episode_end_idx,
                'length': episode_end_idx - episode_start_idx
            })
    
    # Convert lists to numpy arrays for efficient processing
    for cam_name in camera_names:
        all_images[cam_name] = np.array(all_images[cam_name])
        print(f"Total {cam_name} images: {len(all_images[cam_name])}")
    
    return all_images, image_episode_info

# ========================================
# IMAGE PREPROCESSING FUNCTIONS
# ========================================

def preprocess_image_batch(images, target_size=(480, 640)):
    """Convert numpy images to PyTorch tensors with ACT-style normalization"""
    if images.ndim == 3:  # Handle single image
        images = images[np.newaxis, ...]
    
    batch_size = images.shape[0]
    preprocessed = []
    
    # ACT uses ImageNet normalization
    preprocess = transforms.Compose([
        transforms.Resize(target_size),
        transforms.ToTensor(),  # ✅ This automatically converts HWC → CHW!
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    for i in range(batch_size):
        img = images[i]
        if isinstance(img, np.ndarray):
            img = Image.fromarray(img)
        preprocessed.append(preprocess(img))
    
    return torch.stack(preprocessed)

# ========================================
# BACKBONE SETUP FUNCTION
# ========================================

def setup_visual_backbone_with_projection(hidden_dim=256, position_embedding='sine'):
    """Create ACT-style ResNet34 backbone with input projection"""
    backbone = ACTBackbone(
        backbone_name='resnet34',
        train_backbone=True,
        return_interm_layers=False,
        position_embedding_type=position_embedding,
        hidden_dim=hidden_dim
    )
    
    # ✅ Create input projection layer (512 → 256 for ResNet34)
    input_proj = ACTInputProjection(
        backbone_channels=backbone.num_channels,  # 512 for ResNet34
        hidden_dim=hidden_dim  # 256
    )
    
    return backbone, input_proj

# ========================================
# FEATURE EXTRACTION FUNCTIONS
# ========================================

def extract_visual_features_act_style(backbone, input_proj, images, device):
    """Extract features for a single batch of images with input projection"""
    backbone = backbone.to(device)
    input_proj = input_proj.to(device)
    backbone.eval()
    input_proj.eval()
    images = images.to(device)
    
    with torch.no_grad():
        # Step 1: Extract backbone features
        features, pos_embeds = backbone(images)
        
        # Step 2: Apply input projection (ACT's crucial step)
        projected_features = {}
        for name, feat in features.items():
            projected_features[name] = input_proj(feat)
    
    return projected_features, pos_embeds

def extract_all_visual_features_act_style(backbone, input_proj, all_images, camera_names, device, batch_size=32):
    """
    MAIN FUNCTION: Extract visual features from ALL images following ACT's approach
    
    This replicates what ACT does in detr_vae.py:
    features, pos = self.backbones[0](image[:, cam_id])
    features = features[0]  # take the last layer feature
    pos = pos[0]
    all_cam_features.append(self.input_proj(features))  # ✅ INPUT PROJECTION
    """
    backbone = backbone.to(device)
    input_proj = input_proj.to(device)
    backbone.eval()
    input_proj.eval()
    
    print(f"Extracting features for {len(camera_names)} cameras...")
    print(f"Backbone channels: {backbone.num_channels} → Hidden dim: {input_proj.input_proj.out_channels}")
    
    # Storage for all extracted features
    all_features = {cam_name: [] for cam_name in camera_names}
    all_pos_embeds = {cam_name: [] for cam_name in camera_names}
    
    total_images = len(all_images[camera_names[0]])
    
    with torch.no_grad():
        # Process each camera separately (ACT processes cameras independently)
        for cam_name in camera_names:
            print(f"Processing {cam_name} camera ({total_images} images)...")
            
            cam_features = []
            cam_pos_embeds = []
            
            # Process in batches to avoid GPU memory overflow
            for i in range(0, total_images, batch_size):
                end_idx = min(i + batch_size, total_images)
                batch_images = all_images[cam_name][i:end_idx]
                
                # Convert numpy images to PyTorch tensors
                batch_tensor = preprocess_image_batch(batch_images)
                
                # Extract features with projection (mimics ACT's complete pipeline)
                features_dict, pos_dict = extract_visual_features_act_style(backbone, input_proj, batch_tensor, device)
                
                # Take the last layer features (mimics ACT's features[0] and pos[0])
                features = features_dict['0']  # [batch, 256, H, W] AFTER projection
                pos = pos_dict['0']           # [batch, 256, H, W] position embeddings
                
                # Store on CPU to save GPU memory
                cam_features.append(features.cpu())
                cam_pos_embeds.append(pos.cpu())
                
                # Progress indicator
                if (i // batch_size + 1) % 10 == 0:
                    print(f"  Processed batch {i//batch_size + 1}/{(total_images-1)//batch_size + 1}")
            
            # Combine all batches for this camera
            all_features[cam_name] = torch.cat(cam_features, dim=0)
            all_pos_embeds[cam_name] = torch.cat(cam_pos_embeds, dim=0)
            
            print(f"  ✅ {cam_name} features shape: {all_features[cam_name].shape}")
            print(f"  ✅ {cam_name} pos embeds shape: {all_pos_embeds[cam_name].shape}")
    
    return all_features, all_pos_embeds

def create_act_style_visual_inputs(all_features, all_pos_embeds, camera_names, indices, device):
    """
    Create visual inputs for training batch (mimics ACT's forward pass)
    
    This replicates the camera loop in ACT's detr_vae.py:
    for cam_id, cam_name in enumerate(self.camera_names):
        features, pos = self.backbones[0](image[:, cam_id])
        all_cam_features.append(self.input_proj(features))  # ✅ Already projected!
        all_cam_pos.append(pos)
    """
    all_cam_features = []
    all_cam_pos = []
    
    for cam_id, cam_name in enumerate(camera_names):
        # Get features for specific image indices
        features = all_features[cam_name][indices].to(device)      # [batch, 256, H, W] (already projected!)
        pos = all_pos_embeds[cam_name][indices].to(device)         # [batch, 256, H, W] position embeddings
        
        # ✅ Features are already projected (512 → 256), ready for transformer
        all_cam_features.append(features)
        all_cam_pos.append(pos)
    
    return all_cam_features, all_cam_pos

# ========================================
# MAIN EXECUTION
# ========================================

# Load all images from episodes
camera_names = ['front']
all_images, image_episode_info = load_all_images_from_episodes(data_dir, camera_names)

if all_images is not None:
    # Display loading summary
    print(f"\nImage loading summary:")
    for cam_name in camera_names:
        print(f"{cam_name} camera: {len(all_images[cam_name])} total images")
        print(f"Image shape: {all_images[cam_name][0].shape}")
    
    print(f"\nEpisode boundaries:")
    for info in image_episode_info:
        print(f"Episode {info['episode_idx']} ({info['filename']}): images {info['start_idx']}-{info['end_idx']-1} ({info['length']} images)")
    
    # ========================================
    # BACKBONE + PROJECTION SETUP
    # ========================================
    print(f"\n{'='*40}")
    print("SETTING UP BACKBONE WITH INPUT PROJECTION")
    print(f"{'='*40}")
    
    backbone, input_proj = setup_visual_backbone_with_projection(hidden_dim=256, position_embedding='sine')
    print(f"Backbone channels: {backbone.num_channels}")
    print(f"Input projection: {backbone.num_channels} → {input_proj.input_proj.out_channels}")
    
    # ========================================
    # SINGLE IMAGE TEST
    # ========================================
    print(f"\n{'='*40}")
    print("TESTING SINGLE IMAGE PROCESSING WITH PROJECTION")
    print(f"{'='*40}")
    
    print(f"Testing with image shape: {all_images['front'][0].shape}")
    test_image = preprocess_image_batch(all_images['front'][0])
    print(f"Preprocessed image shape: {test_image.shape}")
    
    features, pos_embeds = extract_visual_features_act_style(backbone, input_proj, test_image, device)
    
    print(f"\nSingle image features extracted (with projection):")
    for name, feat in features.items():
        print(f"Feature '{name}': {feat.shape} (projected)")
        print(f"Position embedding '{name}': {pos_embeds[name].shape}")
    
    # ========================================
    # ALL IMAGES FEATURE EXTRACTION WITH PROJECTION
    # ========================================
    print(f"\n{'='*60}")
    print("EXTRACTING VISUAL FEATURES FOR ALL IMAGES (WITH PROJECTION)")
    print(f"{'='*60}")
    
    # Extract features for all images (this is the main operation)
    all_features, all_pos_embeds = extract_all_visual_features_act_style(
        backbone, input_proj, all_images, camera_names, device, batch_size=16
    )
    
    print(f"\n✅ Feature extraction with projection completed!")
    print(f"Features extracted for {len(camera_names)} cameras")
    for cam_name in camera_names:
        print(f"  {cam_name}: {all_features[cam_name].shape} (projected to 256 channels)")
    
    # ========================================
    # TEST BATCH CREATION (for training)
    # ========================================
    print(f"\n{'='*40}")
    print("TESTING BATCH CREATION FOR TRAINING")
    print(f"{'='*40}")
    
    # Sample some images for batch processing test
    test_indices = [0, 150, 300, 450]
    test_indices = [idx for idx in test_indices if idx < len(all_images[camera_names[0]])]
    
    all_cam_features, all_cam_pos = create_act_style_visual_inputs(
        all_features, all_pos_embeds, camera_names, test_indices, device
    )
    
    print(f"ACT-style batch inputs created (with projection):")
    for cam_id, cam_name in enumerate(camera_names):
        print(f"  Camera {cam_id} ({cam_name}):")
        print(f"    Features: {all_cam_features[cam_id].shape} (projected)")
        print(f"    Position: {all_cam_pos[cam_id].shape}")
    
    # Demonstrate ACT's concatenation approach
    if len(camera_names) > 1:
        # Multiple cameras: concatenate along width dimension
        src = torch.cat(all_cam_features, dim=3)
        pos = torch.cat(all_cam_pos, dim=3)
        print(f"  Concatenated src: {src.shape}")
        print(f"  Concatenated pos: {pos.shape}")
    else:
        # Single camera: use directly
        src = all_cam_features[0]
        pos = all_cam_pos[0]
        print(f"  Single camera src: {src.shape}")
        print(f"  Single camera pos: {pos.shape}")
    
    # ✅ Show the critical dimension transformation
    print(f"\n{'='*50}")
    print("ACT DIMENSION TRANSFORMATION SUMMARY")
    print(f"{'='*50}")
    print(f"Raw backbone features: [batch, 512, H, W]")
    print(f"After input_proj:      [batch, 256, H, W]")
    print(f"Position embeddings:   [batch, 256, H, W]")
    print(f"Ready for transformer: ✅")
    
    # Memory usage report
    if torch.cuda.is_available():
        print(f"\nGPU Memory Usage:")
        print(f"  Allocated: {torch.cuda.memory_allocated(device)/1024**3:.2f} GB")
        print(f"  Cached: {torch.cuda.memory_reserved(device)/1024**3:.2f} GB")
    
    print(f"\n✅ All visual features extracted, projected, and ready for ACT training!")

else:
    print("❌ No images loaded!")

# 2 Model Definition
## 2.1 Position Encoding Layer

In [ ]:
def _get_sinusoid_encoding_table(self, n_position, d_hid):
        """ACT-style sinusoidal positional encoding"""
        def get_position_angle_vec(position):
            return [position / np.power(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)]

        sinusoid_table = np.array([get_position_angle_vec(pos_i) for pos_i in range(n_position)])
        sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # dim 2i
        sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # dim 2i+1

        return torch.FloatTensor(sinusoid_table).unsqueeze(0)

## 2.2 VAE , encoder_input = [CLS, current_qpos, action_sequence]
encoder_input = torch.cat([cls_embed, qpos_embed, action_embed], axis=1)
ACT's terminology:
"query model" = encoder that processes [CLS + qpos + actions]
It "queries" the combined information to extract latent representation

In [ ]:
import sys
sys.path.append(r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\detr\models')
from transformer import build_transformer

class TransformerArgs:
    """Arguments object for ACT transformer configuration"""
    def __init__(self, hidden_dim=256, dropout=0.1, nheads=8, dim_feedforward=2048, 
                 enc_layers=6, dec_layers=6, pre_norm=False):
        self.hidden_dim = hidden_dim           # ✅ d_model parameter
        self.dropout = dropout                 # ✅ dropout rate
        self.nheads = nheads                  # ✅ number of attention heads
        self.dim_feedforward = dim_feedforward # ✅ feedforward dimension
        self.enc_layers = enc_layers          # ✅ encoder layers
        self.dec_layers = dec_layers          # ✅ decoder layers
        self.pre_norm = pre_norm              # ✅ normalize_before

class ActionChunkingVAEWithACTInput(nn.Module):
    def __init__(self, 
                 input_dim=6,        # 6 joint positions (current robot state)
                 output_dim=6,       # 6 joint actions (robot commands)
                 chunk_size=10,      # Predict next 10 actions (action horizon)
                 d_model=256,        # Embedding dimension (matches visual features)
                 latent_dim=256,      # Size of latent space Z
                 nhead=8,            # Multi-head attention heads
                 num_layers=6,       # Transformer layers
                 dropout=0.1):
        super().__init__()
        
        # Store key parameters
        self.chunk_size = chunk_size      # How many future actions to predict
        self.output_dim = output_dim      # Dimension of each action (6 joints)
        self.d_model = d_model            # Hidden dimension throughout model
        self.latent_dim = latent_dim      # Dimension of latent variable Z
        
        # ========================================
        # PROJECTION LAYERS: Convert inputs to common embedding space (d_model=256)
        # ========================================
        
        # Project joint positions (6D) to embedding space (256D)
        self.encoder_joint_proj = nn.Linear(input_dim, d_model)    # 6 → 256 (current robot state)
        
        # Project action sequences (6D) to embedding space (256D) 
        self.encoder_action_proj = nn.Linear(output_dim, d_model)  # 6 → 256 (demonstrated actions)
        
        # Learnable [CLS] token - used to aggregate global information
        self.cls_embed = nn.Embedding(1, d_model)                 # Special token for global context
        
        # ========================================
        # VISUAL FEATURE PROCESSING
        # ========================================
        
        # Project visual features to match embedding dimension if needed
        # Note: Visual features from backbone are already 256D after input_proj
        # via input_proj in the backbone setup, so we don't need additional projection here
        # self.visual_proj = nn.Linear(d_model, d_model)  # 256 → 256 (optional transformation)
        
        # Positional encoding for visual features (spatial positions)
        self.visual_pos_proj = nn.Linear(d_model, d_model)  # For visual position embeddings
        
        # ========================================
        # POSITIONAL ENCODING: Helps model understand sequence order, For encoder sequence (CLS + qpos + actions)
        # ========================================
        
        # ACT's sinusoidal positional encoding for sequence tokens
        # Sequence structure: [CLS] + [qpos] + [action_0] + [action_1] + ... + [action_N-1]
        # Total length: 1 (CLS) + 1 (qpos) + chunk_size (actions) = 12 tokens
        self.register_buffer('pos_table', self._get_sinusoid_encoding_table(1 + 1 + chunk_size, d_model))
        
        # ========================================
        # TRANSFORMER ENCODER: Processes concatenated sequence to extract Z
        # ========================================
        
        # ACT's "query model" - actually an encoder that processes:
        # [CLS] + [current_qpos] + [demonstrated_actions] → latent representation Z
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,                # 256D embeddings
            nhead=nhead,                    # 8 attention heads
            dim_feedforward=2048,           # ACT uses 2048 for feedforward
            dropout=dropout,                # Regularization
            activation='relu',              # ACT uses ReLU
            batch_first=False               # Sequence-first format [seq, batch, dim]
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # ========================================
        # VAE COMPONENTS: Learn latent variable Z
        # ========================================
        
        # Project CLS output to VAE parameters (mu and logvar)
        # CLS token (256D) → mu (64D) + logvar (64D) = 128D total
        self.latent_proj = nn.Linear(d_model, latent_dim * 2)  # 256 → 512 (mu + logvar)
        
        # Project sampled latent Z back to embedding space for decoder
        self.latent_out_proj = nn.Linear(latent_dim, d_model)  # 256 → 256 (Z → embedding)
        
        # ========================================
        # ACT'S CUSTOM TRANSFORMER: Replaces standard decoder
        # ========================================
        
        # ✅ CREATE ARGS OBJECT FOR ACT TRANSFORMER
        transformer_args = TransformerArgs(
            hidden_dim=d_model,             # ✅ 256 (matches args.hidden_dim)
            dropout=dropout,                # ✅ 0.1 (matches args.dropout)
            nheads=nhead,                   # ✅ 8 (matches args.nheads)
            dim_feedforward=2048,           # ✅ 2048 (matches args.dim_feedforward)
            enc_layers=num_layers,          # ✅ 6 (matches args.enc_layers)
            dec_layers=num_layers,          # ✅ 6 (matches args.dec_layers)
            pre_norm=False                  # ✅ False (matches args.pre_norm)
        )
        
        # ✅ BUILD ACT'S CUSTOM TRANSFORMER WITH ARGS OBJECT
        # This handles the multimodal fusion of visual + latent + proprioceptive inputs
        self.transformer = build_transformer(transformer_args)
        
        # ✅ LEARNABLE QUERY EMBEDDINGS: For action prediction
        # These are the "questions" that the transformer will answer with actions
        # Each query corresponds to one future action timestep
        self.query_embed = nn.Embedding(chunk_size, d_model)  # 10 × 256 (action queries)
        
        # ✅ ADDITIONAL POSITION EMBEDDINGS: For multimodal inputs
        # Index 0: position embedding for proprioception (current joint state)
        # Index 1: position embedding for latent variable Z (behavioral pattern)
        self.additional_pos_embed = nn.Embedding(2, d_model)  # 2 × 256
        
        # ✅ INPUT PROJECTION FOR PROPRIOCEPTION
        # This matches ACT's input_proj_robot_state - projects joint positions for custom transformer
        self.input_proj_robot_state = nn.Linear(input_dim, d_model)  # 6 → 256
        
        # ========================================
        # OUTPUT LAYERS: Convert transformer output to actions
        # ========================================
        
        # Final prediction layer: transformer output (256D) → action (6D)
        self.action_head = nn.Linear(d_model, output_dim)  # 256 → 6 (final actions)
        
        return
        
    def forward(self, current_qpos, visual_features=None, visual_pos=None, action_sequence=None, is_pad=None):
        """
        ACT Forward Pass with Visual Features
        
        Args:
            current_qpos: [batch, 6] - Current robot joint positions
            visual_features: [batch, 256, H, W] - Projected visual features from backbone
            visual_pos: [batch, 256, H, W] - Visual position embeddings
            action_sequence: [batch, chunk_size, 6] - Demonstrated action sequence (training only)
            is_pad: [batch, chunk_size] - Padding mask for actions
            
        Returns:
            During training: action_pred, mu, logvar, latent_sample
            During inference: action_pred
        """
        batch_size = current_qpos.size(0)
        is_training = action_sequence is not None
        
        # ========================================
        # STEP 1: ENCODE TO LATENT SPACE Z (Training)
        # ========================================
        
        if is_training:
            # Project demonstrated actions to embedding space
            # [batch, chunk_size, 6] → [batch, chunk_size, 256]
            action_embed = self.encoder_action_proj(action_sequence)
        
            # Project current joint positions to embedding space
            # [batch, 6] → [batch, 256] → [batch, 1, 256]
            qpos_embed = self.encoder_joint_proj(current_qpos)
            qpos_embed = qpos_embed.unsqueeze(1)  # Add sequence dimension
            
            # Get learnable CLS token for global context aggregation
            # [1, 256] → [batch, 1, 256]
            cls_embed = self.cls_embed.weight.unsqueeze(0).repeat(batch_size, 1, 1)
            
            # ✅ CORE ACT SEQUENCE: Concatenate [CLS] + [current_qpos] + [action_sequence]
            # This creates the input that encoder will process to learn behavioral patterns
            encoder_input = torch.cat([cls_embed, qpos_embed, action_embed], dim=1)  # [batch, seq+2, 256]
            encoder_input = encoder_input.permute(1, 0, 2)  # [seq+2, batch, 256] (transformer format)
            
            # Handle padding mask for variable-length sequences
            if is_pad is not None:
                # CLS and qpos tokens are never padding
                cls_joint_is_pad = torch.full((batch_size, 2), False).to(current_qpos.device)
                is_pad = torch.cat([cls_joint_is_pad, is_pad], dim=1)  # [batch, seq+2]
            
            # Add positional encoding to help model understand sequence order
            pos_embed = self.pos_table.clone().detach()
            pos_embed = pos_embed.permute(1, 0, 2)  # [seq+2, 1, 256]
            
            # ✅ TRANSFORMER ENCODER: Process sequence to extract behavioral patterns
            # Input: [CLS] + [qpos] + [actions] with positional encoding
            # Output: Contextualized representations, especially CLS token
            encoder_output = self.encoder(encoder_input + pos_embed, src_key_padding_mask=is_pad)
            
            # Extract CLS token output - contains global behavioral information
            cls_output = encoder_output[0]  # [batch, 256] - CLS token representation
            
            # ✅ VAE ENCODING: Convert CLS output to latent variable Z
            # This captures the behavioral pattern in a compact latent space
            latent_info = self.latent_proj(cls_output)  # [batch, 256] → [batch, 128]
            mu = latent_info[:, :self.latent_dim]      # [batch, 64] - mean
            logvar = latent_info[:, self.latent_dim:]  # [batch, 64] - log variance
            
            # Sample latent variable Z using reparameterization trick
            latent_sample = self.reparameterize(mu, logvar)  # [batch, 64] - This is Z!
            
        # ========================================
        # STEP 2: DECODE ACTIONS USING CUSTOM TRANSFORMER
        # ========================================
        
        # For inference, we can use a zero or learned latent
        if not is_training:
            latent_sample = torch.zeros(batch_size, self.latent_dim).to(current_qpos.device)
            mu = logvar = None

        # 1. Project latent Z back to embedding space
        latent_input = self.latent_out_proj(latent_sample)  # [batch, 64] → [batch, 256]
        
        # 2. Project proprioception (current joint positions)
        proprio_input = self.input_proj_robot_state(current_qpos)  # [batch, 6] → [batch, 256]
        
        # 3. Handle visual features and position embeddings
        if visual_features is not None and visual_pos is not None:
            # Visual features are already projected by backbone's input_proj
            # Shape: [batch, 256, H, W] - ready for custom transformer
            src = visual_features      # [batch, 256, H, W] - visual features
            pos_embed = visual_pos     # [batch, 256, H, W] - visual position embeddings
            mask = None                # No masking for visual features
        else:
            # Handle case where visual features are not provided
            print("⚠️ Warning: No visual features provided, using dummy features")
            batch_size = current_qpos.size(0)
            src = torch.zeros(batch_size, self.d_model, 15, 20).to(current_qpos.device)
            pos_embed = torch.zeros(batch_size, self.d_model, 15, 20).to(current_qpos.device)
            mask = None
        
        # ✅ CALL ACT'S CUSTOM TRANSFORMER
        # This is the exact same call as in detr_vae.py line 126:
        # hs = self.transformer(src, None, self.query_embed.weight, pos, latent_input, proprio_input, self.additional_pos_embed.weight)[0]
        
        transformer_output = self.transformer(
            src=src,                                      # [batch, 256, H, W] - visual features
            mask=mask,                                    # None - no masking
            query_embed=self.query_embed.weight,          # [chunk_size, 256] - learnable action queries
            pos_embed=pos_embed,                          # [batch, 256, H, W] - visual position embeds
            latent_input=latent_input,                    # [batch, 256] - projected latent Z
            proprio_input=proprio_input,                  # [batch, 256] - projected joint positions
            additional_pos_embed=self.additional_pos_embed.weight  # [2, 256] - pos embeds for latent+proprio
        )
        
        # Extract the decoder output (first element of returned tuple)
        decoder_output = transformer_output[0]  # [chunk_size, batch, 256]
        #print(f"DEBUG - decoder_output shape before action_head: {decoder_output.shape}")
        
        # ✅ GENERATE ACTION PREDICTIONS: Convert decoder output to joint commands
        action_pred = self.action_head(decoder_output)  # [chunk_size, batch, 6]
        #print(f"DEBUG - action_pred shape before permute: {action_pred.shape}")
        
        # ✅ ENSURE CORRECT OUTPUT FORMAT: [batch, chunk_size, 6]
        if action_pred.shape[0] != batch_size:
        # If first dimension is not batch_size, we need to permute
            action_pred = action_pred.permute(1, 0, 2)  # [batch, chunk_size, 6]
            #print(f"DEBUG - action_pred shape after permute: {action_pred.shape}")
        else:
            #print(f"DEBUG - action_pred shape (already correct): {action_pred.shape}")
            pass



        # Return based on training/inference mode
        if is_training:
            return action_pred, mu, logvar, latent_sample
        else:
            return action_pred
            
    def reparameterize(self, mu, logvar):
        """
        VAE Reparameterization Trick
        
        During training: Sample Z ~ N(mu, sigma) where sigma = exp(logvar/2)
        During inference: Use mean Z = mu (deterministic)
        """
        if self.training:
            std = (logvar / 2).exp()  # Convert log variance to standard deviation
            eps = torch.randn_like(std)  # Sample noise from standard normal
            return mu + std * eps  # Reparameterized sample: Z = mu + sigma * epsilon
        else:
            return mu  # Deterministic inference
    
    def _get_sinusoid_encoding_table(self, n_position, d_hid):
        """
        Generate sinusoidal positional encoding table
        
        Args:
            n_position: Maximum sequence length (12 for ACT: 1 CLS + 1 qpos + 10 actions)
            d_hid: Hidden dimension (256)
            
        Returns:
            pos_table: [1, n_position, d_hid] positional encoding table
        """
        def get_position_angle_vec(position):
            return [position / np.power(10000, 2 * (hid_j // 2) / d_hid) for hid_j in range(d_hid)]

        sinusoid_table = np.array([get_position_angle_vec(pos_i) for pos_i in range(n_position)])
        sinusoid_table[:, 0::2] = np.sin(sinusoid_table[:, 0::2])  # Even dimensions: sine
        sinusoid_table[:, 1::2] = np.cos(sinusoid_table[:, 1::2])  # Odd dimensions: cosine

        return torch.FloatTensor(sinusoid_table).unsqueeze(0)  # [1, n_position, d_hid]
        
    def init_weights(self):
        """Initialize model weights with small random values"""
        initrange = 0.1
        self.encoder_joint_proj.weight.data.uniform_(-initrange, initrange)
        self.encoder_action_proj.weight.data.uniform_(-initrange, initrange)
        self.cls_embed.weight.data.uniform_(-initrange, initrange)
        self.latent_proj.weight.data.uniform_(-initrange, initrange)
        self.latent_out_proj.weight.data.uniform_(-initrange, initrange)
        self.action_head.weight.data.uniform_(-initrange, initrange)
                
        # Initialize query embeddings and additional position embeddings
        self.query_embed.weight.data.uniform_(-initrange, initrange)
        self.additional_pos_embed.weight.data.uniform_(-initrange, initrange)

In [ ]:
def create_act_training_data_official_style(qpos_df, action_df, all_images, camera_names, 
                                          chunk_size=10, train_split=0.8, image_episode_info=None):
    """
    Create ACT training data that matches OFFICIAL implementation
    
    Key difference: Store RAW IMAGES, not pre-extracted features
    Visual processing happens during forward pass (like official ACT)
    """
    
    # Extract joint columns
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    qpos_data = qpos_df[joint_cols].values
    action_data = action_df[joint_cols].values
    
    print(f"Creating ACT training data (Official Style):")
    print(f"QPos data shape: {qpos_data.shape}")
    print(f"Action data shape: {action_data.shape}")
    print(f"Images available: {len(all_images[camera_names[0]])}")
    print(f"Chunk size: {chunk_size}")
    
    # Normalize data
    qpos_min, qpos_max = qpos_data.min(axis=0), qpos_data.max(axis=0)
    qpos_normalized = (qpos_data - qpos_min) / (qpos_max - qpos_min + 1e-8)
    
    action_min, action_max = action_data.min(axis=0), action_data.max(axis=0)
    action_normalized = (action_data - action_min) / (action_max - action_min + 1e-8)
    
    # ========================================
    # CREATE SEQUENCES WITH RAW IMAGES (Official ACT Style)
    # ========================================
    
    sequences = []
    total_timesteps = len(qpos_data)
    total_images = len(all_images[camera_names[0]])
    
    # Ensure alignment
    if total_timesteps != total_images:
        print(f"⚠️ Timestep/image mismatch: {total_timesteps} vs {total_images}")
        total_available = min(total_timesteps, total_images)
    else:
        total_available = total_timesteps
        print(f"✅ Perfect alignment: {total_available} timesteps")
    
    # Process episodes with proper boundaries
    if image_episode_info:
        for episode_info in image_episode_info:
            episode_start = episode_info['start_idx']
            episode_end = min(episode_info['end_idx'], total_available)
            
            print(f"Processing episode {episode_info['episode_idx']}: {episode_start}-{episode_end-1}")
            
            # Create sequences within episode (don't cross boundaries)
            for t in range(episode_start, episode_end - chunk_size):
                
                # ✅ STORE RAW DATA (like official ACT)
                current_qpos = qpos_normalized[t]                    # [6] - current state
                action_sequence = action_normalized[t:t+chunk_size]  # [chunk_size, 6] - future actions
                
                # ✅ STORE RAW IMAGES (not pre-extracted features)
                batch_images = {}
                for cam_name in camera_names:
                    # Store the actual image (will be processed during forward pass)
                    batch_images[cam_name] = all_images[cam_name][t]  # [H, W, 3] - raw image
                
                sequence = {
                    'current_qpos': current_qpos,                # [6] - normalized qpos
                    'action_sequence': action_sequence,          # [chunk_size, 6] - normalized actions
                    'images': batch_images,                      # Dict of raw images per camera
                    'timestep': t,                              # For debugging
                    'episode_idx': episode_info['episode_idx']   # Episode ID
                }
                
                sequences.append(sequence)
    
    else:
        # Fallback: treat as single episode
        for t in range(total_available - chunk_size):
            current_qpos = qpos_normalized[t]
            action_sequence = action_normalized[t:t+chunk_size]
            
            batch_images = {}
            for cam_name in camera_names:
                batch_images[cam_name] = all_images[cam_name][t]
            
            sequence = {
                'current_qpos': current_qpos,
                'action_sequence': action_sequence,
                'images': batch_images,
                'timestep': t,
                'episode_idx': 0
            }
            
            sequences.append(sequence)
    
    print(f"✅ Created {len(sequences)} training sequences")
    
    # Train/test split
    split_idx = int(len(sequences) * train_split)
    train_sequences = sequences[:split_idx]
    test_sequences = sequences[split_idx:]
    
    # Convert to PyTorch format
    train_data = []
    for seq in train_sequences:
        train_data.append({
            'current_qpos': torch.FloatTensor(seq['current_qpos']),
            'action_sequence': torch.FloatTensor(seq['action_sequence']),
            'images': seq['images'],  # Keep as raw images
            'timestep': seq['timestep']
        })
    
    test_data = []
    for seq in test_sequences:
        test_data.append({
            'current_qpos': torch.FloatTensor(seq['current_qpos']),
            'action_sequence': torch.FloatTensor(seq['action_sequence']),
            'images': seq['images'],  # Keep as raw images
            'timestep': seq['timestep']
        })
    
    normalization_params = {
        'qpos_min': qpos_min, 'qpos_max': qpos_max,
        'action_min': action_min, 'action_max': action_max
    }
    
    print(f"\n✅ Official ACT Data Preparation Complete:")
    print(f"Training sequences: {len(train_data)}")
    print(f"Test sequences: {len(test_data)}")
    print(f"Images stored as: Raw numpy arrays (processed during forward pass)")
    
    return train_data, test_data, normalization_params

def get_act_batch_official_style(data, start_idx, batch_size, backbone, input_proj, device):
    """
    Create batches with REAL-TIME visual processing (Official ACT Style)
    
    This matches the official ACT forward pass:
    features, pos = self.backbones[0](image[:, cam_id])
    """
    
    end_idx = min(start_idx + batch_size, len(data))
    batch_data = data[start_idx:end_idx]
    actual_batch_size = len(batch_data)
    
    # Extract components
    batch_current_qpos = torch.stack([sample['current_qpos'] for sample in batch_data], dim=0)
    batch_action_sequence = torch.stack([sample['action_sequence'] for sample in batch_data], dim=0)
    
    # ✅ DEBUG: Print shapes to identify the issue
    #print(f"DEBUG - Batch size: {actual_batch_size}")
    #print(f"DEBUG - batch_current_qpos shape: {batch_current_qpos.shape}")
    #print(f"DEBUG - batch_action_sequence shape: {batch_action_sequence.shape}")
    #print(f"DEBUG - Expected action shape: [batch_size, chunk_size, 6] = [{actual_batch_size}, 10, 6]")
    
    # ✅ PROCESS IMAGES REAL-TIME (like official ACT)
    camera_names = list(batch_data[0]['images'].keys())
    all_cam_features = []
    all_cam_pos = []
    
    backbone.eval()
    input_proj.eval()
    
    with torch.no_grad():
        for cam_name in camera_names:
            # Collect batch of images for this camera
            cam_images = []
            for sample in batch_data:
                img = sample['images'][cam_name]  # [H, W, 3] numpy
                img_tensor = preprocess_image_batch(img)  # [1, 3, H, W]
                cam_images.append(img_tensor[0])  # [3, H, W]
            
            # Stack into batch: [batch, 3, H, W]
            batch_cam_images = torch.stack(cam_images, dim=0).to(device)
            
            # ✅ OFFICIAL ACT PROCESSING: features, pos = self.backbones[0](image[:, cam_id])
            features, pos = backbone(batch_cam_images)
            features = features['0']  # Take last layer features
            pos = pos['0']           # Take corresponding position embeddings
            
            # ✅ OFFICIAL ACT PROJECTION: self.input_proj(features)
            projected_features = input_proj(features)
            
            all_cam_features.append(projected_features)  # [batch, 256, H, W]
            all_cam_pos.append(pos)                     # [batch, 256, H, W]
    
    # ✅ OFFICIAL ACT CONCATENATION: fold camera dimension into width
    if len(camera_names) > 1:
        src = torch.cat(all_cam_features, dim=3)  # Concatenate along width
        pos_embed = torch.cat(all_cam_pos, dim=3)
    else:
        src = all_cam_features[0]
        pos_embed = all_cam_pos[0]
    
    # ✅ FIX: Create padding mask with correct dimensions
    chunk_size = batch_action_sequence.shape[1]  # Should be 10
    batch_is_pad = torch.full((actual_batch_size, chunk_size), False, dtype=torch.bool)
    
    # ✅ DEBUG: Print final shapes before returning
    #print(f"DEBUG - Final src shape: {src.shape}")
    #print(f"DEBUG - Final pos_embed shape: {pos_embed.shape}")
    #print(f"DEBUG - Final batch_is_pad shape: {batch_is_pad.shape}")

    # Move to device
    batch_current_qpos = batch_current_qpos.to(device)
    batch_action_sequence = batch_action_sequence.to(device)
    batch_is_pad = batch_is_pad.to(device)
    
    return batch_current_qpos, src, pos_embed, batch_action_sequence, batch_is_pad



In [ ]:
chunk_size = 10

train_data, test_data, norm_params = create_act_training_data_official_style(
    qpos_df, action_df, all_images, camera_names,
    chunk_size=chunk_size, train_split=0.8, image_episode_info=image_episode_info
)

## Training phase

In [ ]:
def act_loss_function(predictions, targets, mu, logvar, kl_weight=0.0001):
    """
    ACT Loss: L1 Reconstruction + KL Divergence
    """
    # L1 reconstruction loss
    l1_loss = F.l1_loss(predictions, targets)
    
    # KL divergence loss (VAE regularization)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)
    
    # Combined loss
    total_loss = l1_loss + kl_weight * kl_loss
    
    return total_loss, l1_loss, kl_loss

In [ ]:
def train_act_model_official_style(model, backbone, input_proj, train_data, test_data, num_epochs=100):
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    
    for epoch in range(num_epochs):
        model.train()
        
        for batch_idx in range(0, len(train_data), batch_size):
            # ✅ Real-time visual processing (like official ACT)
            batch_qpos, batch_visual_feat, batch_visual_pos, batch_actions, batch_pad = get_act_batch_official_style(
                train_data, batch_idx, batch_size, backbone, input_proj, device
            )
            
            # ✅ Forward pass (now matches official ACT exactly)
            action_pred, mu, logvar, latent_sample = model(
                current_qpos=batch_qpos,
                visual_features=batch_visual_feat,  # Real-time processed
                visual_pos=batch_visual_pos,        # Real-time processed
                action_sequence=batch_actions,
                is_pad=batch_pad
            )
            
            # Compute loss
            total_loss, l1_loss, kl_loss = act_loss_function(
                action_pred, batch_actions, mu, logvar, kl_weight
            )
            
            # Backward pass
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            
            epoch_loss += total_loss.item()
        
        # Print progress
        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Loss = {epoch_loss:.4f}")
    
    return model

In [ ]:
def train_act_model(model, backbone, input_proj, train_data, test_data, num_epochs=100, batch_size=4, kl_weight=0.0001, 
                   lr=1e-4, save_freq=50, checkpoint_dir=None, device=None):
    """
    Complete ACT Training Loop with Real-Time Visual Processing
    
    This implements the exact training procedure from the official ACT paper:
    - Real-time visual processing during training (like official ACT)
    - L1 reconstruction loss for action prediction accuracy
    - KL divergence loss for VAE regularization
    - AdamW optimizer with learning rate scheduling
    - Validation monitoring to prevent overfitting
    - NO EARLY STOPPING (as recommended for ACT)
    
    Args:
        model: ActionChunkingVAEWithACTInput model
        backbone: ACT backbone for visual feature extraction
        input_proj: ACT input projection layer
        train_data: List of training samples with raw images
        test_data: List of validation samples with raw images
        num_epochs: Number of training epochs (default: 100)
        batch_size: Training batch size (default: 4, good for visual features)
        kl_weight: Weight for KL divergence loss (default: 0.0001 from ACT paper)
        lr: Learning rate (default: 1e-4 from ACT paper)
        save_freq: Save checkpoint every N epochs (default: 50)
        checkpoint_dir: Directory to save checkpoints (optional)
        device: torch.device for training (auto-detect if None)
        
    Returns:
        model: Trained model
        training_history: Dict with loss history for plotting
    """
    
    # ========================================
    # DEVICE SETUP - CRITICAL FOR GPU TRAINING
    # ========================================
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # ✅ ENSURE DEVICE IS SPECIFIC (e.g., cuda:0 instead of cuda)
    if device.type == 'cuda' and device.index is None:
        device = torch.device('cuda:0')
    
    print(f"🔧 Using device: {device}")
    
    # ✅ ENSURE ALL MODELS ARE ON CORRECT DEVICE
    model = model.to(device)
    backbone = backbone.to(device)
    input_proj = input_proj.to(device)
    
    print(f"✅ Model moved to {device}")
    print(f"✅ Backbone moved to {device}")
    print(f"✅ Input projection moved to {device}")
    
    # Check if model parameters are on GPU
    model_device = next(model.parameters()).device
    print(f"✅ Model parameters are on: {model_device}")
    
    # ========================================
    # SETUP OPTIMIZER AND SCHEDULER (ACT Configuration)
    # ========================================
    
    # AdamW optimizer - same as official ACT implementation
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=lr,                    # 1e-4 learning rate
        weight_decay=1e-4         # L2 regularization for better generalization
    )
    
    # Learning rate scheduler - cosine annealing (optional but helps convergence)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=num_epochs,         # Full cosine cycle over all epochs
        eta_min=lr * 0.1          # Minimum LR = 10% of initial LR
    )
    
    # ========================================
    # TRAINING HISTORY TRACKING
    # ========================================
    
    training_history = {
        'train_total_loss': [],   # Combined L1 + KL loss
        'train_l1_loss': [],      # Action reconstruction loss
        'train_kl_loss': [],      # VAE regularization loss
        'val_total_loss': [],     # Validation losses
        'val_l1_loss': [],
        'val_kl_loss': [],
        'learning_rates': []      # Track LR changes
    }
    
    print(f"{'='*60}")
    print(f"STARTING ACT TRAINING (OFFICIAL STYLE)")
    print(f"{'='*60}")
    print(f"Training samples: {len(train_data)}")
    print(f"Validation samples: {len(test_data)}")
    print(f"Batch size: {batch_size}")
    print(f"Epochs: {num_epochs}")
    print(f"Learning rate: {lr}")
    print(f"KL weight: {kl_weight}")
    print(f"Device: {device}")
    print(f"Real-time visual processing: ✅")
    print(f"{'='*60}")
    
    # Track best validation loss for model saving (but no early stopping)
    best_val_loss = float('inf')
    
    # ========================================
    # MAIN TRAINING LOOP - NO EARLY STOPPING
    # ========================================
    
    for epoch in range(num_epochs):
        
        # =====================================
        # TRAINING PHASE
        # =====================================
        
        model.train()  # Enable dropout and batch norm training mode
        epoch_train_loss = 0.0
        epoch_train_l1 = 0.0
        epoch_train_kl = 0.0
        num_train_batches = 0
        
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Iterate through training data in batches
        for batch_idx in range(0, len(train_data), batch_size):
            
            # =====================================
            # PREPARE TRAINING BATCH WITH REAL-TIME VISUAL PROCESSING
            # =====================================
            
            # ✅ REAL-TIME VISUAL PROCESSING (like official ACT)
            batch_qpos, batch_visual_feat, batch_visual_pos, batch_actions, batch_pad = get_act_batch_official_style(
                train_data, batch_idx, batch_size, backbone, input_proj, device  # ✅ All required parameters
            )
            
            # ✅ IMPROVED DEVICE CHECKS (handle cuda:0 vs cuda)
            def check_device_match(tensor, expected_device, tensor_name):
                if tensor.device.type != expected_device.type:
                    raise AssertionError(f"{tensor_name} on {tensor.device}, expected {expected_device}")
                if expected_device.type == 'cuda' and tensor.device.index != expected_device.index:
                    raise AssertionError(f"{tensor_name} on {tensor.device}, expected {expected_device}")
            
            check_device_match(batch_qpos, device, "batch_qpos")
            check_device_match(batch_visual_feat, device, "batch_visual_feat")
            check_device_match(batch_visual_pos, device, "batch_visual_pos")
            check_device_match(batch_actions, device, "batch_actions")
            check_device_match(batch_pad, device, "batch_pad")
            
            # =====================================
            # FORWARD PASS - ACT MODEL INFERENCE
            # =====================================
            
            # Call model forward pass with all required inputs
            # This replicates the exact ACT forward pass from detr_vae.py
            action_pred, mu, logvar, latent_sample = model(
                current_qpos=batch_qpos,           # [batch, 6] - current robot state
                visual_features=batch_visual_feat,  # [batch, 256, H, W] - projected visual features (real-time)
                visual_pos=batch_visual_pos,       # [batch, 256, H, W] - spatial position embeddings (real-time)
                action_sequence=batch_actions,     # [batch, chunk_size, 6] - target action sequence
                is_pad=batch_pad                   # [batch, chunk_size] - padding mask
            )
            
            # ✅ VERIFY OUTPUT TENSORS ARE ON CORRECT DEVICE
            check_device_match(action_pred, device, "action_pred")
            check_device_match(mu, device, "mu")
            check_device_match(logvar, device, "logvar")
            
            #print(f"DEBUG - action_pred shape: {action_pred.shape}")
            #print(f"DEBUG - batch_actions shape: {batch_actions.shape}")
            #print(f"DEBUG - mu shape: {mu.shape}")
            #print(f"DEBUG - logvar shape: {logvar.shape}")

            # =====================================
            # LOSS COMPUTATION - ACT LOSS FUNCTION
            # =====================================
            
            # Compute ACT's combined loss: L1 reconstruction + KL divergence
            total_loss, l1_loss, kl_loss = act_loss_function(
                predictions=action_pred,    # [batch, chunk_size, 6] - predicted actions
                targets=batch_actions,      # [batch, chunk_size, 6] - ground truth actions
                mu=mu,                      # [batch, latent_dim] - VAE mean
                logvar=logvar,             # [batch, latent_dim] - VAE log variance
                kl_weight=kl_weight        # Weight for KL regularization
            )
            
            # ✅ VERIFY LOSS TENSORS ARE ON CORRECT DEVICE
            check_device_match(total_loss, device, "total_loss")
            
            # =====================================
            # BACKWARD PASS - GRADIENT COMPUTATION
            # =====================================
            
            # Zero gradients from previous iteration
            optimizer.zero_grad()
            
            # Compute gradients via backpropagation
            total_loss.backward()
            
            # Gradient clipping to prevent exploding gradients (common in transformers)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update model parameters
            optimizer.step()
            
            # =====================================
            # ACCUMULATE TRAINING METRICS
            # =====================================
            
            epoch_train_loss += total_loss.item()
            epoch_train_l1 += l1_loss.item()
            epoch_train_kl += kl_loss.item()
            num_train_batches += 1
            
            # Print batch progress for long epochs
            #if num_train_batches % 20 == 0:
                #print(f"  Batch {num_train_batches}: Loss = {total_loss.item():.4f} "
                      #f"(L1: {l1_loss.item():.4f}, KL: {kl_loss.item():.6f})")
                
                # Memory usage monitoring for GPU
                # if device.type == 'cuda':
                    #print(f"    GPU Memory: {torch.cuda.memory_allocated(device)/1024**3:.2f}GB / "
                          #f"{torch.cuda.memory_reserved(device)/1024**3:.2f}GB")
        
        # =====================================
        # VALIDATION PHASE
        # =====================================
        
        model.eval()  # Disable dropout, enable evaluation mode
        epoch_val_loss = 0.0
        epoch_val_l1 = 0.0
        epoch_val_kl = 0.0
        num_val_batches = 0
        
        with torch.no_grad():  # Disable gradient computation for efficiency
            for batch_idx in range(0, len(test_data), batch_size):
                
                # ✅ REAL-TIME VISUAL PROCESSING FOR VALIDATION
                batch_qpos, batch_visual_feat, batch_visual_pos, batch_actions, batch_pad = get_act_batch_official_style(
                    test_data, batch_idx, batch_size, backbone, input_proj, device  # ✅ All required parameters
                )
                
                # Forward pass (no gradient computation)
                action_pred, mu, logvar, latent_sample = model(
                    current_qpos=batch_qpos,
                    visual_features=batch_visual_feat,
                    visual_pos=batch_visual_pos,
                    action_sequence=batch_actions,
                    is_pad=batch_pad
                )
                
                # Compute validation loss
                total_loss, l1_loss, kl_loss = act_loss_function(
                    action_pred, batch_actions, mu, logvar, kl_weight
                )
                
                # Accumulate validation metrics
                epoch_val_loss += total_loss.item()
                epoch_val_l1 += l1_loss.item()
                epoch_val_kl += kl_loss.item()
                num_val_batches += 1
        
        # =====================================
        # COMPUTE EPOCH AVERAGES
        # =====================================
        
        avg_train_loss = epoch_train_loss / num_train_batches
        avg_train_l1 = epoch_train_l1 / num_train_batches
        avg_train_kl = epoch_train_kl / num_train_batches
        
        avg_val_loss = epoch_val_loss / num_val_batches if num_val_batches > 0 else 0
        avg_val_l1 = epoch_val_l1 / num_val_batches if num_val_batches > 0 else 0
        avg_val_kl = epoch_val_kl / num_val_batches if num_val_batches > 0 else 0
        
        # =====================================
        # UPDATE LEARNING RATE
        # =====================================
        
        scheduler.step()  # Update learning rate according to schedule
        
        # =====================================
        # RECORD TRAINING HISTORY
        # =====================================
        
        training_history['train_total_loss'].append(avg_train_loss)
        training_history['train_l1_loss'].append(avg_train_l1)
        training_history['train_kl_loss'].append(avg_train_kl)
        training_history['val_total_loss'].append(avg_val_loss)
        training_history['val_l1_loss'].append(avg_val_l1)
        training_history['val_kl_loss'].append(avg_val_kl)
        training_history['learning_rates'].append(optimizer.param_groups[0]['lr'])
        
        # =====================================
        # PROGRESS REPORTING
        # =====================================
        
        print(f"  Train Loss: {avg_train_loss:.4f} (L1: {avg_train_l1:.4f}, KL: {avg_train_kl:.6f})")
        print(f"  Val Loss:   {avg_val_loss:.4f} (L1: {avg_val_l1:.4f}, KL: {avg_val_kl:.6f})")
        
        # =====================================
        # MODEL CHECKPOINTING (NO EARLY STOPPING)
        # =====================================
        
        # Save best model based on validation loss (but keep training)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            if checkpoint_dir:
                best_model_path = f"{checkpoint_dir}/best_act_model.pth"
                torch.save({
                    'epoch': epoch + 1,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'best_val_loss': best_val_loss,
                    'training_history': training_history
                }, best_model_path)
                print(f"  ✅ New best model saved! Val Loss: {best_val_loss:.4f}")
        
        # Save periodic checkpoints
        if checkpoint_dir and (epoch + 1) % save_freq == 0:
            checkpoint_path = f"{checkpoint_dir}/act_model_epoch_{epoch+1}.pth"
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'training_history': training_history
            }, checkpoint_path)
            print(f"  💾 Checkpoint saved: {checkpoint_path}")
    
    print(f"\n{'='*60}")
    print(f"TRAINING COMPLETED!")
    print(f"{'='*60}")
    print(f"Best validation loss: {best_val_loss:.4f}")
    print(f"Final learning rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    return model, training_history

In [ ]:
# ========================================
# SETUP: Create Model and Visual Components
# ========================================

# Ensure device is explicitly set
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Training device: {device}")

# Create model
chunk_size = 10
model = ActionChunkingVAEWithACTInput(
    input_dim=6,
    output_dim=6,
    chunk_size=chunk_size,
    d_model=256,
    latent_dim=256,
    nhead=8,
    num_layers=6,
    dropout=0.1
)

# Setup backbone and input projection (already created earlier)
# backbone, input_proj = setup_visual_backbone_with_projection(hidden_dim=256, position_embedding='sine')

# Move models to device
model = model.to(device)
backbone = backbone.to(device)
input_proj = input_proj.to(device)

# Create checkpoint directory
import os
checkpoint_dir = "act_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

# ========================================
# START TRAINING WITH REAL-TIME VISUAL PROCESSING
# ========================================

print("🚀 Starting ACT training with real-time visual processing...")
trained_model, history = train_act_model(
    model=model,
    backbone=backbone,                # ✅ Required for real-time visual processing
    input_proj=input_proj,           # ✅ Required for feature projection
    train_data=train_data,
    test_data=test_data,
    num_epochs=4000,                 # Train for full epochs (no early stopping)
    batch_size=32,                    # Good for visual features
    kl_weight=100,               # ACT paper value
    lr=5e-5,                        # ACT paper value
    save_freq=50,                   # Save every 50 epochs
    checkpoint_dir=checkpoint_dir,
    device=device                   # ✅ Explicit device
)

print("✅ Training completed successfully!")